# Which ensemble detects faults at a low false-alarm rate?

Start with predictions from the six seed-42 models. Recreate the 57 ensembles, the fault counts in Table V, the machine breakdown in Table VI and the retrospective split check. All outputs below are computed from the score archive.

## Scores from the six models

The five CNNs and XGBoost scored the same clips at seed 42. The saved five-CNN ensemble provides a reference for checking the calculation.

In [1]:
from pathlib import Path
import os
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebooks/code").is_dir() and (p / "results/fits.json").is_file())
os.chdir(root)
import csv
import hashlib
from IPython.display import Markdown, display
import itertools
import json

import numpy as np
from sklearn.metrics import roc_auc_score


NAMES = ("MobileNetV3-Small", "ResNet-18", "EfficientNet-B0", "Compact CNN", "Wide CNN", "XGBoost")
CNN_PARAMS = (1521668, 11172292, 4012096, 177724, 1286964, 0)
MACHINES = ("fan", "pump", "slider", "valve")
evidence = root / "results"
output = root / "runs/ensemble_subsets_seed42"
output.mkdir(exist_ok=True, parents=True)
fits = {r["model"]: r["key"] for r in json.loads((evidence/"fits.json").read_text()) if r["seed"] == 42}
with np.load(evidence/"clip_scores.npz", allow_pickle=False) as archive:
    data = {key: archive[key] for key in archive.files}
a = np.stack([data[fits[n]+"_train"] for n in NAMES])
b = np.stack([data[fits[n]+"_test"] for n in NAMES])
if not (np.isfinite(a).all() and np.isfinite(b).all() and a.min() >= 0 and b.min() >= 0):
    raise ValueError("Invalid input anomaly scores")
a, b = (np.log(np.maximum(values, np.finfo(float).tiny)) for values in (a, b))
y = data["test_label"]
assert not data["train_label"].any() and len(y) == 18714 and y.sum() == 9900

## Calibrate on healthy reference recordings

Standardize log scores separately for each model, machine type and machine ID. The same calibration applies at each noise level.

In [2]:
groups, cells = [], []
for m in MACHINES:
    for mid in np.unique(data["test_id"]):
        tr = np.flatnonzero((data["train_machine"] == m) & (data["train_id"] == mid))
        te = np.flatnonzero((data["test_machine"] == m) & (data["test_id"] == mid))
        groups.append((tr, te))
        for snr in (-6, 0, 6):
            cells.append(te[data["test_snr"][te] == snr])
        mu, sd = a[:, tr].mean(axis=1, keepdims=True), a[:, tr].std(axis=1, keepdims=True)
        assert (sd > 0).all()
        a[:, tr] = (a[:, tr] - mu) / sd
        b[:, te] = (b[:, te] - mu) / sd

## Partition by source recording

A deterministic hash assigns the three noise versions of each source to the same half. We use the first half for exploratory selection and the second half to check the selected configuration.

In [3]:
# Deterministic source-level partition prevents SNR siblings from straddling halves.
sources = [str(m)+"/"+p.split("/", 1)[1] for m,p in zip(data["test_machine"],data["test_path"])]
select = np.array([hashlib.sha256(("subset-study-v1/"+s).encode()).digest()[0] < 128 for s in sources])
all_rows = np.ones(len(y), dtype=bool)
assert len(set(sources))==6238 and not set(np.array(sources)[select]) & set(np.array(sources)[~select])

## Define the metrics

Calculate ROC-AUC and standardized partial AUC within each type, ID and noise group. Calculate recall and false-alarm rate from clip counts at the training-derived threshold.

In [4]:
def summary(y, scores, alarm, cells, keep):
    auc, pauc = [], []
    for cell in cells:
        idx = cell[keep[cell]]
        auc.append(roc_auc_score(y[idx], scores[idx]))
        pauc.append(roc_auc_score(y[idx], scores[idx], max_fpr=0.1))
    tp = int(np.sum(alarm & keep & (y == 1)))
    fp = int(np.sum(alarm & keep & (y == 0)))
    faulty, healthy = int(sum(y[keep])), int(sum(keep) - sum(y[keep]))
    return dict(auc=np.mean(auc), pauc=np.mean(pauc), recall=tp/faulty, fpr=fp/healthy,
                tp=tp, fn=faulty-tp, fp=fp, tn=healthy-fp)

def write_csv(path, rows):
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=rows[0])
        writer.writeheader()
        writer.writerows(rows)

## Evaluate every combination

Average standardized scores with equal weights. Refit the 95th-percentile threshold using only the fused healthy training scores for each type and ID.

In [5]:
rows, selection, check_scores, by_type = [], [], {}, []
for size in range(1, len(NAMES)+1):
    for members in itertools.combinations(range(len(NAMES)), size):
        name = " + ".join(NAMES[i] for i in members)
        header = dict(members=name, size=size, cnn_parameters=sum(CNN_PARAMS[i] for i in members),
                      includes_xgboost=5 in members)
        train, test = a[list(members)].mean(axis=0), b[list(members)].mean(axis=0)
        if members == tuple(range(5)):
            for split, values in [("train", train), ("test", test)]:
                np.testing.assert_allclose(values, data[fits["CNN ensemble"]+"_"+split], atol=1e-11, rtol=1e-11)
        alarm = np.zeros(y.shape, dtype=bool)
        for tr, te in groups:
            alarm[te] = test[te] > np.quantile(train[tr], .95)
        rows.append({**header, **summary(y, test, alarm, cells, all_rows)})
        for i, m in enumerate(MACHINES):
            by_type.append({**header, "machine":m, **summary(y,test,alarm,cells[12*i:12*i+12],data["test_machine"]==m)})
        if size > 1:
            selection.append({**header, **summary(y, test, alarm, cells, select)})
            check_scores[name] = test, alarm

## Check the selected configuration and save results

Choose the first-half leader by partial AUC, breaking ties by ensemble size and name. The output includes all combinations, per-machine results and the grouped check.

In [6]:
best = sorted(selection, key=lambda r: (-r["pauc"],r["size"],r["members"]))[0]
recommended = best["members"]
holdout = {name: summary(y, *check_scores[name], cells, ~select)
           for name in {recommended, " + ".join(NAMES[:5]), " + ".join(NAMES)}}
rows.sort(key=lambda r: -r["pauc"])
write_csv(output/"all_combinations.csv", rows)
write_csv(output/"by_machine.csv", by_type)
study = dict(seed=42, selection_half_clips=int(select.sum()), check_half_clips=int((~select).sum()),
             selected_by_first_half_pauc=best, second_half_results=holdout,
             split="sha256(subset-study-v1/ + machine/source_recording_key), first byte < 128 selects first half",
             caveat="Both halves belong to the project's previously inspected test set.",
             evidence_sha256=hashlib.sha256((evidence/"clip_scores.npz").read_bytes()).hexdigest())
(output/"split_check.json").write_text(json.dumps(study,indent=2)+"\n")

1626

## What does another model buy at the alarm threshold?

Compare the report’s five ensemble configurations on the same clips. The highest-pAUC combination includes ResNet-18, EfficientNet-B0, the wide CNN and XGBoost.

In [7]:
def show(items, label):
    lines = ["| Model or group | AUC (%) | pAUC (%) | TP | FN | FP | TN | Accuracy (%) | Recall (%) | FPR (%) |",
             "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
    for r in items:
        total = sum(r[k] for k in ("tp", "fp", "tn", "fn"))
        accuracy = 100 * (r["tp"] + r["tn"]) / total
        lines.append(f"| {r[label]} | {100*r['auc']:.2f} | {100*r['pauc']:.2f} | {r['tp']} | {r['fn']} | {r['fp']} | {r['tn']} | "
                     f"{accuracy:.2f} | {100*r['recall']:.2f} | {100*r['fpr']:.2f} |")
    display(Markdown("\n".join(lines)))

configs = [[0,1,2,3,4], [1,2,5], [1,2,4,5], [0,1,2,5], list(range(6))]
lookup = {r["members"]: r for r in rows}
report_rows = [lookup[" + ".join(NAMES[i] for i in indices)] for indices in configs]
show(report_rows, "members")

| Model or group | AUC (%) | pAUC (%) | TP | FN | FP | TN | Accuracy (%) | Recall (%) | FPR (%) |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| MobileNetV3-Small + ResNet-18 + EfficientNet-B0 + Compact CNN + Wide CNN | 93.62 | 88.79 | 7592 | 2308 | 720 | 8094 | 83.82 | 76.69 | 8.17 |
| ResNet-18 + EfficientNet-B0 + XGBoost | 95.58 | 90.08 | 7985 | 1915 | 662 | 8152 | 86.23 | 80.66 | 7.51 |
| ResNet-18 + EfficientNet-B0 + Wide CNN + XGBoost | 95.66 | 90.55 | 7989 | 1911 | 654 | 8160 | 86.29 | 80.70 | 7.42 |
| MobileNetV3-Small + ResNet-18 + EfficientNet-B0 + XGBoost | 94.83 | 89.31 | 8042 | 1858 | 777 | 8037 | 85.92 | 81.23 | 8.82 |
| MobileNetV3-Small + ResNet-18 + EfficientNet-B0 + Compact CNN + Wide CNN + XGBoost | 94.97 | 89.90 | 7948 | 1952 | 746 | 8068 | 85.58 | 80.28 | 8.46 |

## Which machines account for the missed faults?

Table VI uses one alarm per clip. The fan and pump have lower recall than the slider and valve. The machine AUCs still average across IDs and noise levels.

In [8]:
winner = rows[0]["members"]
show([r for r in by_type if r["members"] == winner], "machine")

| Model or group | AUC (%) | pAUC (%) | TP | FN | FP | TN | Accuracy (%) | Recall (%) | FPR (%) |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| fan | 90.34 | 83.32 | 3007 | 1418 | 173 | 2269 | 76.83 | 67.95 | 7.08 |
| pump | 93.92 | 85.90 | 953 | 415 | 168 | 2079 | 83.87 | 69.66 | 7.48 |
| slider | 98.84 | 95.13 | 2606 | 64 | 144 | 1770 | 95.46 | 97.60 | 7.52 |
| valve | 99.55 | 97.86 | 1423 | 14 | 169 | 2042 | 94.98 | 99.03 | 7.64 |

## Which clips changed the decision?

Compare alarm decisions clip by clip against the five-CNN rule. Then split the healthy false alarms by noise level. This reproduces the paired comparison and noise result in the discussion.

In [9]:
winning_alarm = check_scores[winner][1]
five_cnn_alarm = check_scores[" + ".join(NAMES[:5])][1]
gained = winning_alarm & ~five_cnn_alarm
lost = five_cnn_alarm & ~winning_alarm
print(f"Faults recovered: {sum(gained & (y == 1))}; previously caught faults lost: {sum(lost & (y == 1))}")
print(f"False alarms avoided: {sum(lost & (y == 0))}; new false alarms: {sum(gained & (y == 0))}")
for level in (-6, 0, 6):
    mask = data["test_snr"] == level
    false_alarms = winning_alarm[mask & (y == 0)]
    caught = winning_alarm[mask & (y == 1)]
    print(f"{level:+} dB: {false_alarms.sum()} false alarms; FPR {100 * false_alarms.mean():.2f}%, "
          f"fault recall {100 * caught.mean():.2f}%")

Faults recovered: 659; previously caught faults lost: 262
False alarms avoided: 235; new false alarms: 169
-6 dB: 514 false alarms; FPR 17.49%, fault recall 83.30%
+0 dB: 73 false alarms; FPR 2.48%, fault recall 76.42%
+6 dB: 67 false alarms; FPR 2.28%, fault recall 82.36%


## Does the selection hold on a second group of recordings?

Rank ensembles on the first half by pAUC, then evaluate that choice on the second half. Both halves came from the previously inspected test set. The three noise mixtures of any one source stay together.

In [10]:
print(f"Selection: {select.sum():,} clips; check: {(~select).sum():,} clips")
print(f"First-half selection: {recommended} ({100*best['pauc']:.2f}% pAUC)")
show([dict(members=name, **holdout[name]) for name in sorted(holdout)], "members")

Selection: 9,276 clips; check: 9,438 clips
First-half selection: ResNet-18 + EfficientNet-B0 + Wide CNN + XGBoost (90.85% pAUC)


| Model or group | AUC (%) | pAUC (%) | TP | FN | FP | TN | Accuracy (%) | Recall (%) | FPR (%) |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| MobileNetV3-Small + ResNet-18 + EfficientNet-B0 + Compact CNN + Wide CNN | 93.43 | 88.48 | 3854 | 1162 | 357 | 4065 | 83.91 | 76.83 | 8.07 |
| MobileNetV3-Small + ResNet-18 + EfficientNet-B0 + Compact CNN + Wide CNN + XGBoost | 94.81 | 89.55 | 4036 | 980 | 369 | 4053 | 85.71 | 80.46 | 8.34 |
| ResNet-18 + EfficientNet-B0 + Wide CNN + XGBoost | 95.65 | 90.42 | 4048 | 968 | 329 | 4093 | 86.26 | 80.70 | 7.44 |